# 01 · Carga de datos y EDA inicial

**Proyecto:** ChurnLens · Diplomado MLDS — Universidad Nacional de Colombia

**Fase TDSP:** 1 · *Entendimiento del negocio y carga de datos* (10 %)

**Objetivo de este notebook:**

1. Demostrar de manera reproducible la **carga** del dataset _Telco Customer Churn_ usando el paquete `churnlens`.
2. Validar el dataset contra el **esquema Pandera** declarado en `src/churnlens/data/schema.py`.
3. Realizar una **primera exploración** numérica y visual: balance del target, distribuciones, missing values, dependencias entre variables.
4. Generar las primeras **hipótesis** sobre _drivers_ de _churn_ que se profundizarán en la Fase 2.

## 0. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Asegura que el paquete sea importable cuando se ejecuta desde la carpeta notebooks/
PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from churnlens.data.loader import TelcoChurnLoader
from churnlens.data.validators import compute_quality_report

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("Python:", sys.version.split()[0])
print("Project root:", PROJECT_ROOT)

## 1. Descarga y carga del dataset

El loader del paquete encapsula el ciclo completo: descarga → hash → casteo → validación Pandera → parquet.

In [ ]:
loader = TelcoChurnLoader()
csv_path = loader.download()
df = loader.load_validated()

summary = loader.summary()
print(f"Filas         : {summary.n_rows:,}")
print(f"Columnas      : {summary.n_cols}")
print(f"Tasa churn    : {summary.target_pos_rate:.4f}")
print(f"NaN TotalCh.  : {summary.n_missing_total_charges}")
print(f"MD5           : {summary.md5}")
print(f"Tamaño (bytes): {summary.bytes:,}")

In [ ]:
df.head()

In [ ]:
df.info()

## 2. Reporte rápido de calidad

In [ ]:
report = compute_quality_report(df)
print(f"Filas duplicadas         : {report.n_duplicates}")
print(f"Columnas constantes      : {report.constant_columns}")
print(f"Tasa de positivos        : {report.target_positive_rate:.4f}")
print(f"Imbalance ratio          : {report.imbalance_ratio:.2f}")
print(f"Faltantes por columna    : {report.missing_per_column}")

## 3. Balance del target

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df["Churn"].value_counts()
ax.bar(counts.index.astype(str), counts.values, color=["#1f6feb", "#e11d48"])
ax.set_title("Distribución del target — Churn")
ax.set_ylabel("# de clientes")
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,}\n({v / counts.sum():.1%})", ha="center", va="bottom")
plt.tight_layout()
plt.show()

## 4. Distribuciones numéricas

In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_cols):
    sns.histplot(data=df, x=col, hue="Churn", kde=True, bins=30, ax=ax, common_norm=False, stat="density")
    ax.set_title(col)
plt.tight_layout()
plt.show()

**Observación:**

* `tenure` es claramente **bimodal** — gran masa en clientes nuevos (`[0,6]`) y otra masa en clientes muy antiguos (`[60,72]`). Los clientes nuevos están sobre-representados entre los _churners_.
* `MonthlyCharges` muestra una cresta inferior (~ USD 20) que corresponde a clientes con solo servicio telefónico, y una banda alta (USD 65–110) con mayor concentración de _churners_.
* `TotalCharges` arrastra el patrón de `tenure × MonthlyCharges`.

## 5. Tasa de churn por variable categórica clave

In [ ]:
categorical_cols = [
    "Contract",
    "InternetService",
    "PaymentMethod",
    "PaperlessBilling",
    "OnlineSecurity",
    "TechSupport",
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), categorical_cols):
    churn_rate = (
        df.groupby(col, observed=False)["Churn"]
        .apply(lambda s: (s == "Yes").mean())
        .sort_values(ascending=False)
    )
    bars = ax.bar(churn_rate.index.astype(str), churn_rate.values, color="#1f6feb")
    ax.axhline(0.2654, color="#e11d48", linestyle="--", linewidth=1, label="Promedio global (26.5 %)")
    ax.set_title(f"Tasa de churn por {col}")
    ax.set_ylabel("P(Churn=Yes)")
    for b, v in zip(bars, churn_rate.values):
        ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.0%}", ha="center", va="bottom", fontsize=8)
    ax.tick_params(axis="x", rotation=15)
    if ax is axes[0, 0]:
        ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

**Hallazgos preliminares clave:**

* `Contract = Month-to-month` ⇒ tasa de churn ~ 43 % (vs ~ 3 % en `Two year`).
* `InternetService = Fiber optic` ⇒ tasa de churn cercana al 42 %, casi duplicando la media.
* `PaymentMethod = Electronic check` ⇒ tasa de churn ~ 45 %; los métodos automáticos muestran tasas significativamente menores.
* La ausencia de _add-ons_ de seguridad (`OnlineSecurity = No`, `TechSupport = No`) coincide con tasas de churn elevadas.

Todas estas señales se cuantificarán rigurosamente en el EDA de la Fase 2.

## 6. Heatmap de correlación (variables numéricas + target codificado)

In [ ]:
df_num = df[["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]].copy()
df_num["Churn"] = (df["Churn"] == "Yes").astype(int)
corr = df_num.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlación de Pearson — variables numéricas vs Churn")
plt.tight_layout()
plt.show()

## 7. Cierre

Este notebook valida que:

* La **carga reproducible** del dataset funciona end-to-end (descarga + hash + casteo + Pandera + parquet).
* El **dataset está limpio** desde el punto de vista de integridad (sin duplicados, sin columnas constantes, sin valores fuera de dominio).
* El **dataset es modelable**: hay señal abundante en variables como `Contract`, `tenure`, `InternetService`, `PaymentMethod`.
* La hipótesis del [Project Charter](../docs/business_understanding/project_charter.md) es plausible: los _drivers_ esperados (antigüedad, contrato, método de pago, profundidad de adopción) aparecen claramente en los gráficos.

Próximo paso → **Fase 2 · EDA + Preprocesamiento.**